In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout, GlobalMaxPooling1D, LeakyReLU, BatchNormalization
from keras.optimizers import RMSprop, Adam, SGD
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report


In [2]:
# # Mount Google Drive to access the dataset
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# Read the CSV file into a DataFrame
df = pd.read_csv('/content/drive/MyDrive/TA DIAZ/datasetdiaz.csv', usecols=['HS', 'Tweet'])
df

,Tweet,HS
0,cowok usaha lacak perhati gue lantas remeh per...,1
1,41 kadang pikir percaya tuhan jatuh kali kali ...,0
2,ku tau mata sipit lihat,0
3,kaum cebong kafir lihat dongok dungu haha,1
4,deklarasi pilih kepala daerah 2018 aman anti h...,0
...,...,...
11482,orang yahudi kristen muslim be emu kumpul mala...,0
11483,bicara ndasmu congor sekata anjing,1
11484,kasur enak kunyuk,0
11485,bom real mudah deteksi bom kubur dahsyat ledak...,0


In [4]:
# Extract the input features (x) and labels (y)
x = df['Tweet'].values
y = df['HS'].values

In [28]:
x = np.where(pd.isna(x), '', x)
# Inisialisasi dan fit TF-IDF vectorizer dengan parameter yang disesuaikan
vectorizer = TfidfVectorizer(
    use_idf=True,        # Menggunakan IDF
    smooth_idf=False,     # Menggunakan IDF smoothing
    ngram_range=(2, 2),   # Hanya unigram
    max_features=2000,    # Menggunakan lebih banyak fitur
    min_df=20              # Menghapus kata-kata yang terlalu jarang muncul
)

tfidf_vectorizer = vectorizer.fit(x)
# Dapatkan ukuran tokenizer/vocabulary
tokenizer_size = len(tfidf_vectorizer.vocabulary_)
tokenizer_size

194

split

In [29]:
# Function to split the data into training and testing sets
def split_train_test(x, y, tfidf_vectorizer, split):
    # Convert the text features to TF-IDF vectors
    x = np.array(tfidf_vectorizer.transform(x).todense())
    # Reshape the input to have 3 dimensions
    x = x.reshape(x.shape[0], x.shape[1], 1)
    # Split the data into training and testing sets
    x_train, x_test, y_train, y_test = train_test_split(x, y, random_state=25)
    # Determine the number of classes in the labels
    num_classes = np.max(y) + 1
    # Convert the categorical labels to one-hot encoded vectors
    y_train = to_categorical(y_train, num_classes)
    y_test = to_categorical(y_test, num_classes)
    # Return the training and testing data
    return x_train, x_test, y_train, y_test

In [30]:
def cnn_model(x, y, test_size):
    # Split the data into training and testing sets
    x_train, x_test, y_train, y_test = split_train_test(x, y, tfidf_vectorizer, split=test_size)

    # Define the sequential model
    model = Sequential()

    # Add a 1D convolutional layer to the model
    model.add(Conv1D(64, 5, activation='relu', input_shape=(tokenizer_size, 1), padding='same'))

    # Add a 1D max pooling layer to the model
    model.add(MaxPooling1D(5, padding='same'))
    model.add(Dropout(0.3))

    # Flatten the output from the previous layer
    model.add(Flatten())

    # Add a dense layer with ReLU activation
    model.add(Dense(64, activation='relu'))

    # model.add(Dense(512, activation='relu'))

    # Add a dense layer with sigmoid activation
    model.add(Dense(2, activation='sigmoid'))


    # Compile the model with Adam optimizer
    optimizer = Adam(learning_rate=0.0001)
    model.compile(loss='binary_crossentropy', optimizer=optimizer, metrics=['accuracy'])

    # Print a summary of the model architecture
    model.summary()

    # Define early stopping to prevent overfitting
    early_stopping = EarlyStopping(patience=5, restore_best_weights=True)

    # Train the model
    model.fit(x_train, y_train, epochs=35, batch_size=16, verbose=1, validation_data=(x_test, y_test), callbacks=[early_stopping])

    # Evaluasi model
    loss, accuracy = model.evaluate(x_test, y_test)
    print(f"Test Loss: {loss:.4f}")
    print(f"Test Accuracy: {accuracy:.4f}")

    # Make predictions on the testing data
    y_pred = model.predict(x_test)

    # Calculate evaluation metrics
    accuracy = accuracy_score(y_test.argmax(axis=1), y_pred.argmax(axis=1))
    precision = precision_score(y_test.argmax(axis=1), y_pred.argmax(axis=1), average='macro')
    recall = recall_score(y_test.argmax(axis=1), y_pred.argmax(axis=1), average='macro')
    f1 = f1_score(y_test.argmax(axis=1), y_pred.argmax(axis=1), average='macro')
    class_repot = classification_report(y_test.argmax(axis=1), y_pred.argmax(axis=1))

    return accuracy, precision, recall, f1, class_repot

In [31]:
# Number of experiments to run
num_experiments = 5
accuracy_scores = []
precision_scores = []
recall_scores = []
f1_scores = []

# Run multiple experiments
for i in range(num_experiments):
    print(f"Experiment {i+1}")

    # Run the GRU model for each experiment
    accuracy, precision, recall, f1, class_repot = cnn_model(x, y, 0.1)

    accuracy_scores.append(accuracy)
    precision_scores.append(precision)
    recall_scores.append(recall)
    f1_scores.append(f1)

    # Print the evaluation metrics for each experiment
    print(f"Evaluation Metrics - Experiment {i+1}:")
    print(f"Accuracy Score: {accuracy:.4f}")
    print(f"Precision Score: {precision:.4f}")
    print(f"Recall Score: {recall:.4f}")
    print(f"F1 Score: {f1:.4f}")
    print()
    print("Classification Report:")
    print(class_repot)
    print('-------------------------------------------------------------------')
    print()

Experiment 1


/usr/local/lib/python3.11/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_25"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv1d_25 (Conv1D)                   │ (None, 194, 64)             │             384 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling1d_25 (MaxPooling1D)      │ (None, 39, 64)              │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_25 (Dropout)                 │ (None, 39, 64)              │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten_25 (Flatten)                 │ (None, 2496)                │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_50 (Dense)                     │ (None, 64)                  │         159,808 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_51 (Dense)                     │ (None, 2)                   │             130 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 160,322 (626.26 KB)

...

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv1d_29 (Conv1D)                   │ (None, 194, 64)             │             384 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling1d_29 (MaxPooling1D)      │ (None, 39, 64)              │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_29 (Dropout)                 │ (None, 39, 64)              │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten_29 (Flatten)                 │ (None, 2496)                │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_58 (Dense)                     │ (None, 64)                  │         159,808 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_59 (Dense)                     │ (None, 2)                   │             130 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 160,322 (626.26 KB)

 Trainable params: 160,322 (626.26 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/35
539/539 ━━━━━━━━━━━━━━━━━━━━ 7s 9ms/step - accuracy: 0.5937 - loss: 0.6837 - val_accuracy: 0.6166 - val_loss: 0.6614
Epoch 2/35
539/539 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.6375 - loss: 0.6483 - val_accuracy: 0.6309 - val_loss: 0.6456
Epoch 3/35
539/539 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.6414 - loss: 0.6371 - val_accuracy: 0.6354 - val_loss: 0.6384
Epoch 4/35
539/539 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.6486 - loss: 0.6297 - val_accuracy: 0.6393 - val_loss: 0.6353
Epoch 5/35
539/539 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.6631 - loss: 0.6167 - val_accuracy: 0.6407 - val_loss: 0.6322
Epoch 6/35
539/539 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.6592 - loss: 0.6201 - val_accuracy: 0.6494 - val_loss: 0.6280
Epoch 7/35
539/539 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.6692 - loss: 0.6083 - val_accuracy: 0.6501 - val_loss: 0.6248
Epoch 8/35
539/539 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.6685 - loss: 0.6117 - val_accuracy: 0.

In [33]:
# Calculate the average evaluation metrics across all experiments
avg_accuracy = np.mean(accuracy_scores)
avg_precision = np.mean(precision_scores)
avg_recall = np.mean(recall_scores)
avg_f1 = np.mean(f1_scores)

# Calculate the average evaluation metrics across all experiments
print("\nAverage Evaluation Metrics:")
print(f"Average Accuracy Score: {avg_accuracy:.4f}")
print(f"Average Precision Score: {avg_precision:.4f}")
print(f"Average Recall Score: {avg_recall:.4f}")
print(f"Average F1 Score: {avg_f1:.4f}")


Average Evaluation Metrics:
Average Accuracy Score: 0.6568
Average Precision Score: 0.6933
Average Recall Score: 0.6311
Average F1 Score: 0.6140
